## Load dirty bounding-box statistics

Load the per-structure dirty bounding-box statistics and inspect their schema.

In [46]:
from typing import Any


import pickle
from pathlib import Path
import pandas as pd

# Adjust this path if the notebook kernel is not started in the repo root
seg_dir = Path('../data/segmentations')

# Load per-structure stats (single pickle)
with (seg_dir / 'bb_stats_dict_dirty.pkl').open('rb') as f:
    stats = pickle.load(f)  # dict[structure -> dict[stat_name -> value]]

# Turn into a DataFrame: rows = structures, columns = stats
bb_df = pd.DataFrame.from_dict(stats, orient='index')
bb_df.index.name = 'structure'

print(len(bb_df))
# Print the keys (stat names) in the dict for the first column's first row
first_col = bb_df.columns[0]
first_val = bb_df.iloc[0][first_col]
if isinstance(first_val, dict):
    print(f"Columns in dict for column '{first_col}': {list(first_val.keys())}")
else:
    print(f"First column value is not a dict: {first_val}")

152
Columns in dict for column 'spleen': ['avg_x', 'avg_y', 'avg_z', 'med_x', 'med_y', 'med_z', 'min_x', 'min_y', 'min_z', 'max_x', 'max_y', 'max_z']


In [48]:
# Flatten all dict cells from patient x structure table
cells = [
    cell
    for cell in bb_df.to_numpy().ravel()
    if isinstance(cell, dict)
]

mins_x = [d["min_x"] for d in cells if d.get("min_x") is not None]
mins_y = [d["min_y"] for d in cells if d.get("min_y") is not None]
mins_z = [d["min_z"] for d in cells if d.get("min_z") is not None]

maxs_x = [d["max_x"] for d in cells if d.get("max_x") is not None]
maxs_y = [d["max_y"] for d in cells if d.get("max_y") is not None]
maxs_z = [d["max_z"] for d in cells if d.get("max_z") is not None]

print("global min_x:", min(mins_x) if mins_x else None)
print("global min_y:", min(mins_y) if mins_y else None)
print("global min_z:", min(mins_z) if mins_z else None)

print("global max_x:", max(maxs_x) if maxs_x else None)
print("global max_y:", max(maxs_y) if maxs_y else None)
print("global max_z:", max(maxs_z) if maxs_z else None)

global min_x: 1
global min_y: 0
global min_z: 0
global max_x: 255
global max_y: 191
global max_z: 455


## Compute global averages from dirty stats

Convert the dirty bounding-box statistics into a global CoM table `global_avg_xyz`.

In [49]:
# Global average x, y, z per structure
# import numpy as np

# If bb_df is structure x stats (one patient): rows = structures, columns include avg_x, avg_y, avg_z
if all(col in bb_df.columns for col in ['avg_x', 'med_x', 'avg_y', 'med_y', 'avg_z', 'med_z']):
    global_avg_xyz = bb_df[['avg_x', 'med_x', 'avg_y', 'med_y', 'avg_z', 'med_z']].copy()
    global_avg_xyz
else:
    # bb_df is patient x structure: rows = patient IDs, columns = structure names, cell = dict
    def _extract_xyz(cell):
        if pd.isna(cell) or not isinstance(cell, dict):
            return None
        x, y, z, med_x, med_y, med_z = cell.get('avg_x'), cell.get('avg_y'), cell.get('avg_z'), cell.get('med_x'), cell.get('med_y'), cell.get('med_z')
        return (x, med_x, y, med_y, z, med_z) if all(val is not None for val in (x, y, z, med_x, med_y, med_z)) else None

    rows = []
    for struct in bb_df.columns:
        xyz_list = [_extract_xyz(bb_df.loc[pid, struct]) for pid in bb_df.index]
        xyz_list = [x for x in xyz_list if x is not None]
        if not xyz_list:
            continue
        xs, med_xs, ys, med_ys, zs, med_zs = zip(*xyz_list)
        rows.append({
            'structure': struct,
            'avg_x': np.mean(xs),
            'med_x': np.mean(med_xs),
            'avg_y': np.mean(ys),
            'med_y': np.mean(med_ys),
            'avg_z': np.mean(zs),
            'med_z': np.mean(med_zs)
        })

    global_avg_xyz = pd.DataFrame(rows).set_index('structure')
    print(global_avg_xyz)

NameError: name 'np' is not defined

## Explore sorted global averages

Sort structures by their global average CoM along each axis for quick inspection.

In [50]:
# Structures sorted by global average x, y, z

# Ensure global_avg_xyz exists (run previous cells first)

sorted_by_x = global_avg_xyz.sort_values('avg_x')
sorted_by_y = global_avg_xyz.sort_values('avg_y')
sorted_by_z = global_avg_xyz.sort_values('avg_z')

sorted_by_x.head(), sorted_by_y.head(), sorted_by_z.head()

NameError: name 'global_avg_xyz' is not defined

## Compute cleaned (CBR) global averages

Aggregate the color-corrected / cleaned bounding-box statistics into a global CoM table `global_avg_xyz_cbr`.

In [51]:
# Compute global_avg_xyz for cbr_clr_bb_stats_dict.pkl (color-corrected / variant stats)

import pickle
from pathlib import Path
import pandas as pd
import numpy as np

seg_dir = Path('../data/segmentations')

with (seg_dir / 'cbr_clr_bb_stats_dict.pkl').open('rb') as f:
    cbr_stats = pickle.load(f)  # dict[structure -> dict[stat_name -> value]] or patient->structure->stats

# Try structure x stats first
if isinstance(next(iter(cbr_stats.values())), dict) and 'avg_x' in next(iter(cbr_stats.values())):
    cbr_bb_df = pd.DataFrame.from_dict(cbr_stats, orient='index')
    cbr_bb_df.index.name = 'structure'
    global_avg_xyz_cbr = cbr_bb_df[['avg_x', 'avg_y', 'avg_z', 'med_x', 'med_y', 'med_z']].copy()
else:
    # patient x structure -> stats dict (similar to bb_df in patient×structure layout)
    # Build a DataFrame like the other case and aggregate
    rows = []
    for patient_id, struct_map in cbr_stats.items():
        for struct, s in struct_map.items():
            if not isinstance(s, dict):
                continue
            x, y, z, med_x, med_y, med_z = s.get('avg_x'), s.get('avg_y'), s.get('avg_z'), s.get('med_x'), s.get('med_y'), s.get('med_z')
            if x is None or y is None or z is None or med_x is None or med_y is None or med_z is None:
                continue
            rows.append({'patient': patient_id, 'structure': struct, 'avg_x': x, 'avg_y': y, 'avg_z': z, 'med_x': med_x, 'med_y': med_y, 'med_z': med_z})
    tmp = pd.DataFrame(rows)
    global_avg_xyz_cbr = tmp.groupby('structure')[['avg_x', 'avg_y', 'avg_z', 'med_x', 'med_y', 'med_z']].mean()

global_avg_xyz_cbr.head()

,avg_x,avg_y,avg_z,med_x,med_y,med_z
structure,,,,,,
adrenal_gland_left,145.626781,82.440844,328.573639,145.616505,82.519417,328.597087
adrenal_gland_right,113.207797,77.232948,331.178200,113.201342,77.271812,331.167785
aorta,135.742626,82.625078,342.686804,135.802632,82.078947,343.480263
autochthon_left,145.431229,55.780047,321.545247,144.552632,55.427632,318.394737
autochthon_right,110.766355,56.035991,322.199375,111.631579,55.644737,318.664474


## Join original and cleaned global averages

Combine original and cleaned global CoM tables into a single comparison dataframe.

In [52]:
# Compare original and cbr_clr global averages side by side

comparison = global_avg_xyz.join(global_avg_xyz_cbr, how='inner', lsuffix='_orig', rsuffix='_cbr')
comparison

NameError: name 'global_avg_xyz' is not defined

## Largest shifts between dirty and cleaned CoMs

Compute per-axis and total differences between original (dirty) and cleaned (CBR) global CoMs and rank structures by shift size.

In [53]:
# Structures with the largest difference between dirty (orig) and cleaned (cbr) stats

# Absolute differences per axis
comparison['diff_x'] = (comparison['avg_x_orig'] - comparison['avg_x_cbr']).abs()
comparison['diff_y'] = (comparison['avg_y_orig'] - comparison['avg_y_cbr']).abs()
comparison['diff_z'] = (comparison['avg_z_orig'] - comparison['avg_z_cbr']).abs()

# Combined difference (Euclidean distance in xyz-space)
comparison['diff_total'] = (comparison[['diff_x', 'diff_y', 'diff_z']] ** 2).sum(axis=1) ** 0.5

# Top 10 structures with biggest total difference
comparison.sort_values('diff_total', ascending=False)

NameError: name 'comparison' is not defined

## Difference between averages and medians (CBR)

Quantify how far each structure's cleaned average CoM is from its cleaned median CoM.

In [54]:
comparison['diff_avg_med_x'] = (comparison['avg_x_cbr'] - comparison['med_x_cbr'])
comparison['diff_avg_med_y'] = (comparison['avg_y_cbr'] - comparison['med_y_cbr'])
comparison['diff_avg_med_z'] = (comparison['avg_z_cbr'] - comparison['med_z_cbr'])

# Combined difference (Euclidean distance in xyz-space)
comparison['diff_total_avg_med'] = (comparison[['diff_avg_med_x', 'diff_avg_med_y', 'diff_avg_med_z']] ** 2).sum(axis=1) ** 0.5

# Top 10 structures with biggest total difference
comparison[['diff_avg_med_x', 'diff_avg_med_y', 'diff_avg_med_z', 'diff_total_avg_med']].sort_values('diff_total_avg_med', ascending=False)

NameError: name 'comparison' is not defined

## 3D visualization of cleaned CoMs

Visualize the cleaned (CBR) structure averages and medians in 3D using Plotly.

In [55]:
import plotly.graph_objects as go

# Use global_avg_xyz_cbr for cleaned (cbr) means and medians
# The points will be named by structure and color/shape encoded by avg/med

fig = go.Figure()

# Add average points
fig.add_trace(go.Scatter3d(
    x=global_avg_xyz_cbr['avg_x'],
    y=global_avg_xyz_cbr['avg_y'],
    z=global_avg_xyz_cbr['avg_z'],
    mode='markers+text',
    name='Average',
    marker=dict(size=7, color='blue'),
    text=global_avg_xyz_cbr.index,
    textposition='top center',
    hovertemplate=
        "<b>%{text}</b><br>"
        "avg_x: %{x:.2f}<br>"
        "avg_y: %{y:.2f}<br>"
        "avg_z: %{z:.2f}<br>"
))

# Add median points
fig.add_trace(go.Scatter3d(
    x=global_avg_xyz_cbr['med_x'],
    y=global_avg_xyz_cbr['med_y'],
    z=global_avg_xyz_cbr['med_z'],
    mode='markers+text',
    name='Median',
    marker=dict(size=7, color='red', symbol='diamond'),
    text=global_avg_xyz_cbr.index,
    textposition='bottom center',
    hovertemplate=
        "<b>%{text}</b><br>"
        "med_x: %{x:.2f}<br>"
        "med_y: %{y:.2f}<br>"
        "med_z: %{z:.2f}<br>"
))

fig.update_layout(
    title="3D Visualization of Structure Averages and Medians (CBR)",
    legend=dict(itemsizing='constant'),
    scene=dict(
        xaxis_title="Lateral (X)",
        yaxis_title="Anteroposterior (Y)",
        zaxis_title="Vertical (Z)"
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=800, height=600
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## Export cleaned CoM set

Export the cleaned global center-of-mass (CoM) coordinates to JSON for use by the main app.

In [56]:
# Export cleaned global CoM set (cbr_clr) using the format of test_structures.json
# X: lateral axis (right to left)  -> com_lateral
# Z: vertical axis (bottom to top) -> com_vertical

from typing import Any


import json
from pathlib import Path

# Make sure global_avg_xyz_cbr exists (run previous cells first)
output_dir = Path('../data/structures')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'CoM_cleaned_normalized_global_avg_xyz.json'  # used by the main app as a CoM set



structures_list = []
for struct, row in global_avg_xyz_cbr.iterrows():
    structures_list.append({
        "name": struct,
        "com_vertical": float(row["avg_z"]),          # vertical axis (bottom->top)
        "com_lateral": float(row["avg_x"]),           # lateral axis (right->left)
        "com_anteroposterior": float(row["avg_y"]),   # AP axis (back->front)
    })

x, y, z = [], [], []
for item in structures_list:
    x.append(item["com_vertical"])
    y.append(item["com_lateral"])
    z.append(item["com_anteroposterior"])

min_x, max_x = min(x), max(x)
min_y, max_y = min(y), max(y)
min_z, max_z = min(z), max(z)

normalized_5_95_x = [5 + 90 * (v - min_x) / (max_x - min_x) for v in x]
normalized_5_95_y = [5 + 90 * (v - min_y) / (max_y - min_y) for v in y]
normalized_5_95_z = [5 + 90 * (v - min_z) / (max_z - min_z) for v in z]

print(normalized_5_95_x)
print(normalized_5_95_y)
print(normalized_5_95_z)

for i, item in enumerate(structures_list):
    item["com_vertical"] = normalized_5_95_x[i]
    item["com_lateral"] = normalized_5_95_y[i]
    item["com_anteroposterior"] = normalized_5_95_z[i]

payload = {"structures": structures_list}

with output_path.open('w') as f:
    json.dump(payload, f, indent=2)

output_path

[64.9954885960867, 65.69447048998704, 68.78301555765839, 63.10929077656062, 63.284837951457426, 95.0, 59.16412845311821, 62.4854678611169, 73.03046392974277, 29.255510960757604, 29.4924719031071, 5.0, 64.13980637459352, 48.20736155394572, 48.32769357957445, 52.4661650375277, 52.61140749212687, 51.46888257003176, 51.49414207830095, 71.21756377357224, 51.325856278465764, 51.494331032528024, 78.29371744691001, 77.44424378029939, 53.21696992815974, 53.62333128640937, 52.18469591134556, 52.48177675329168, 54.564514095998135, 54.41020869526292, 62.631664618987934, 65.3907456046015, 63.6326216012526, 63.06330603817723, 66.51063691927324, 73.88315440433762, 73.62183563307421, 65.31162317194651, 65.06507973032457, 47.913920260825265, 35.587549803671976, 35.509107169933074, 53.31238372422364, 35.38630163989896, 36.04856728722419, 57.95961195684493, 71.28090353701485, 67.37811290193251, 67.01553937852816, 39.98509820481479, 39.74847864321165, 32.11667547006065, 32.24191451552059, 8.45669934914228

PosixPath('../data/Input_CoM_structures/CoM_cleaned_normalized_global_avg_xyz.json')

X: lateral axis (right to left) same as our
Y: ap axis (back to front)
Z: vertical axis (bottom to top)